# Tratamento da Base de Execução Orçamentária — 2024 e 2025

Este notebook executa o ETL da base de execução orçamentária do Município de São Paulo:

1. extrai os arquivos de 2024 e 2025 por meio de `core/downloads.orcamento.load_orcamento`;
2. valida o esquema de cada exercício;
3. preserva as regras de conversão das colunas monetárias;
4. consolida os dois períodos;
5. exporta e relê o CSV final para validar a carga.

A saída mantém exatamente o layout definido na issue #103.


In [1]:
from datetime import datetime
from hashlib import sha256
import pandas as pd
from os import environ, makedirs, path, replace
from shutil import copy2

from core.downloads.orcamento import load_orcamento


In [2]:
YEARS = [2024, 2025]

OUTPUT_DIR = environ.get(
    "ORCAMENTO_OUTPUT_DIR",
    path.join("data_output", "orcamento"),
)
OUTPUT_FILENAME = environ.get(
    "ORCAMENTO_OUTPUT_FILENAME",
    "orcamento_2024_2025.csv",
)
OUTPUT_PATH = path.join(OUTPUT_DIR, OUTPUT_FILENAME)


In [3]:
EXPECTED_COLUMNS = [
    "Cd_AnoExecucao",
    "Cd_Exercicio",
    "Cd_Dotacao_Id",
    "Administracao",
    "Cd_Orgao",
    "Sigla_Orgao",
    "Ds_Orgao",
    "Cd_Funcao",
    "Ds_Funcao",
    "Cd_SubFuncao",
    "Ds_SubFuncao",
    "Cd_Programa",
    "Ds_Programa",
    "ProjetoAtividade",
    "Ds_Projeto_Atividade",
    "Vl_Orcado_Ano",
    "Vl_Suplementado",
    "Vl_Reduzido",
    "Vl_SuplementadoLiquido",
    "Vl_Orcado_Atualizado",
    "Vl_ReservadoLiquido",
    "Vl_EmpenhadoLiquido",
    "Vl_Liquidado",
    "Vl_Pago",
]

VALUE_COLUMNS = [
    column for column in EXPECTED_COLUMNS
    if column.startswith("Vl_")
]


In [4]:
ZERO_TOKENS = {
    "",
    "-",
    "--",
    "---",
    "NA",
    "N/A",
    "NAN",
    "NULL",
    "NONE",
}


def normalize_numeric_column(series: pd.Series) -> pd.Series:
    """Normaliza a representação textual brasileira sem converter silenciosamente."""
    normalized = series.fillna("").astype(str).str.strip()
    normalized = normalized.str.replace("R$", "", regex=False)
    normalized = normalized.str.replace(" ", "", regex=False)
    normalized = normalized.str.replace("\u00A0", "", regex=False)

    has_thousands_and_decimal = (
        normalized.str.contains(".", regex=False)
        & normalized.str.contains(",", regex=False)
    )
    normalized.loc[has_thousands_and_decimal] = (
        normalized.loc[has_thousands_and_decimal]
        .str.replace(".", "", regex=False)
    )

    normalized = normalized.str.replace(",", ".", regex=False)

    zero_mask = normalized.str.upper().isin(ZERO_TOKENS)
    normalized.loc[zero_mask] = "0"

    return normalized


def invalid_numeric_samples(series: pd.Series, limit: int = 10) -> list[str]:
    """Retorna amostra de valores inválidos antes da conversão."""
    normalized = normalize_numeric_column(series)
    valid_mask = normalized.str.fullmatch(
        r"[+-]?(?:\d+(?:\.\d+)?|\.\d+)",
        na=False,
    )
    return (
        series.loc[~valid_mask]
        .fillna("")
        .astype(str)
        .drop_duplicates()
        .head(limit)
        .tolist()
    )


def parse_numeric_column(series: pd.Series) -> pd.Series:
    """Converte valores monetários e falha explicitamente em valores inválidos."""
    normalized = normalize_numeric_column(series)
    return pd.to_numeric(normalized)


In [5]:
def validate_and_transform(dataframe: pd.DataFrame, year: int) -> pd.DataFrame:
    """Valida uma origem e aplica o tratamento sem alterar a quantidade de linhas."""
    missing_columns = [
        column for column in EXPECTED_COLUMNS
        if column not in dataframe.columns
    ]
    extra_columns = [
        column for column in dataframe.columns
        if column not in EXPECTED_COLUMNS
    ]

    if missing_columns:
        raise ValueError(
            f"Exercício {year}: colunas ausentes: {missing_columns}; "
            f"colunas extras: {extra_columns}. "
            "Impacto: o layout obrigatório de 24 colunas não pode ser produzido."
        )

    print(f"Exercício {year}: colunas extras ignoradas: {extra_columns}")

    for year_column in ["ANO", "Cd_Exercicio", "Cd_AnoExecucao"]:
        if year_column not in dataframe.columns:
            raise ValueError(
                f"Exercício {year}: coluna de origem ausente: {year_column}."
            )
        observed_years = set(
            dataframe[year_column].dropna().astype(str).unique()
        )
        if observed_years != {str(year)}:
            raise ValueError(
                f"Exercício {year}: valores inesperados em {year_column}: "
                f"{sorted(observed_years)}"
            )

    invalid_values = {
        column: samples
        for column in VALUE_COLUMNS
        if (samples := invalid_numeric_samples(dataframe[column]))
    }
    if invalid_values:
        raise ValueError(
            f"Exercício {year}: valores monetários inválidos: "
            f"{invalid_values}"
        )

    rows_before = dataframe.shape[0]
    treated = dataframe[EXPECTED_COLUMNS].copy()
    for column in VALUE_COLUMNS:
        treated[column] = parse_numeric_column(treated[column])

    if treated.shape[0] != rows_before:
        raise AssertionError(
            f"Exercício {year}: quantidade de linhas alterada no tratamento."
        )
    if treated.columns.tolist() != EXPECTED_COLUMNS:
        raise AssertionError(
            f"Exercício {year}: layout final diferente do esperado."
        )
    return treated


numeric_test = pd.Series([
    "1.234,56", "-1.234,56", "-", "", "NULL", "NA", "N/A", "NAN", "NONE"
])
expected_test = [1234.56, -1234.56, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
assert parse_numeric_column(numeric_test).tolist() == expected_test
try:
    parse_numeric_column(pd.Series(["valor inesperado"]))
except ValueError:
    pass
else:
    raise AssertionError("Valor monetário inválido não causou erro.")

synthetic = pd.DataFrame({column: ["0"] for column in EXPECTED_COLUMNS})
synthetic["ANO"] = 2024
synthetic["Cd_Exercicio"] = "2024"
synthetic["Cd_AnoExecucao"] = "2024"
assert validate_and_transform(synthetic, 2024).shape == (1, 24)
try:
    validate_and_transform(synthetic.drop(columns=[EXPECTED_COLUMNS[0]]), 2024)
except ValueError as error:
    assert EXPECTED_COLUMNS[0] in str(error)
else:
    raise AssertionError("Coluna obrigatória ausente não causou erro.")
synthetic_2025 = synthetic.copy()
synthetic_2025[["ANO", "Cd_Exercicio", "Cd_AnoExecucao"]] = 2025
synthetic_consolidated = pd.concat(
    [
        validate_and_transform(synthetic, 2024),
        validate_and_transform(synthetic_2025, 2025),
    ],
    ignore_index=True,
)
assert synthetic_consolidated.shape == (2, 24)
print("Testes sintéticos de transformação aprovados.")


Exercício 2024: colunas extras ignoradas: ['ANO']
Exercício 2024: colunas extras ignoradas: ['ANO']
Exercício 2025: colunas extras ignoradas: ['ANO']
Testes sintéticos de transformação aprovados.


In [6]:
raw_dataframes = {}

for year in YEARS:
    dataframe = load_orcamento(year)
    raw_dataframes[year] = dataframe

    print(
        f"Exercício {year}: "
        f"{dataframe.shape[0]} linhas e "
        f"{dataframe.shape[1]} colunas extraídas."
    )


Exercício 2024: 9344 linhas e 62 colunas extraídas.
Exercício 2025: 8070 linhas e 62 colunas extraídas.


In [7]:
treated_dataframes = []
rows_by_year = {}

for year, dataframe in raw_dataframes.items():
    treated = validate_and_transform(dataframe, year)
    rows_by_year[year] = treated.shape[0]
    treated_dataframes.append(treated)


Exercício 2024: colunas extras ignoradas: ['ANO', '...']
Exercício 2025: colunas extras ignoradas: ['ANO', '...']


In [8]:
df_treated = pd.concat(
    treated_dataframes,
    ignore_index=True,
)

expected_total_rows = sum(rows_by_year.values())

assert df_treated.shape[0] == expected_total_rows, (
    "A quantidade de linhas do consolidado não corresponde "
    "à soma dos exercícios."
)
assert df_treated.columns.tolist() == EXPECTED_COLUMNS, (
    "O consolidado não possui exatamente o layout esperado."
)
observed_exercises = set(df_treated["Cd_Exercicio"].astype(str).unique())
expected_exercises = {str(year) for year in YEARS}
assert observed_exercises == expected_exercises, (
    f"Exercícios inesperados no consolidado: {sorted(observed_exercises)}"
)

duplicate_count = int(df_treated.duplicated().sum())
print(f"Registros completamente duplicados no consolidado: {duplicate_count}")


Registros completamente duplicados no consolidado: 0


In [9]:
exercise_summary = (
    df_treated
    .groupby("Cd_Exercicio", dropna=False)
    .agg(
        quantidade_registros=("Cd_Dotacao_Id", "count"),
        dotacoes_distintas=("Cd_Dotacao_Id", "nunique"),
        orgaos_distintos=("Cd_Orgao", "nunique"),
        total_orcado=("Vl_Orcado_Ano", "sum"),
        total_suplementado=("Vl_Suplementado", "sum"),
        total_reduzido=("Vl_Reduzido", "sum"),
        total_atualizado=("Vl_Orcado_Atualizado", "sum"),
        total_reservado=("Vl_ReservadoLiquido", "sum"),
        total_empenhado=("Vl_EmpenhadoLiquido", "sum"),
        total_liquidado=("Vl_Liquidado", "sum"),
        total_pago=("Vl_Pago", "sum"),
    )
    .reset_index()
)

exercise_summary


  Cd_Exercicio  quantidade_registros  dotacoes_distintas  orgaos_distintos  total_orcado  total_atualizado  total_empenhado  total_liquidado  total_pago
0         2024                  9344                9344                90  111851681558.00   130967285006.33  123909183627.93  117331095186.74  117065381747.14
1         2025                  8070                8070                90  125654200594.00   132985155830.23  123571806694.23  117009214333.13  116798206872.59

In [10]:
def file_sha256(file_path: str) -> str:
    digest = sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


makedirs(OUTPUT_DIR, exist_ok=True)
temporary_output_path = f"{OUTPUT_PATH}.tmp"
if path.exists(temporary_output_path):
    raise FileExistsError(
        f"Arquivo temporário já existe e foi preservado: {temporary_output_path}"
    )

backup_path = None
if path.exists(OUTPUT_PATH):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_dir = path.join(OUTPUT_DIR, "backup")
    makedirs(backup_dir, exist_ok=True)
    filename_stem, filename_extension = path.splitext(OUTPUT_FILENAME)
    backup_path = path.join(
        backup_dir,
        f"{filename_stem}_{timestamp}{filename_extension}",
    )
    original_hash = file_sha256(OUTPUT_PATH)
    copy2(OUTPUT_PATH, backup_path)
    if file_sha256(backup_path) != original_hash:
        raise IOError("O hash do backup difere do arquivo original.")
    print(
        f"Backup criado: {backup_path}; "
        f"tamanho={path.getsize(backup_path)}; sha256={original_hash}"
    )

df_treated.to_csv(
    temporary_output_path,
    index=False,
    sep=";",
    decimal=",",
    encoding="latin1",
)

print(f"Arquivo temporário exportado para: {temporary_output_path}")


Arquivo temporário exportado para: data_output/orcamento/validation/20260730_032231/orcamento_2024_2025.csv.tmp


In [11]:
df_check = pd.read_csv(
    temporary_output_path,
    sep=";",
    decimal=",",
    encoding="latin1",
    dtype=str,
)

assert df_check.shape[0] == df_treated.shape[0], (
    "Quantidade de linhas inconsistente após a exportação."
)
assert df_check.columns.tolist() == EXPECTED_COLUMNS, (
    "Layout inconsistente após a exportação."
)
assert set(df_check["Cd_Exercicio"].astype(str).unique()) == expected_exercises, (
    "O arquivo temporário não contém exatamente os exercícios esperados."
)

temporary_hash = file_sha256(temporary_output_path)
temporary_size = path.getsize(temporary_output_path)
replace(temporary_output_path, OUTPUT_PATH)

df_final_check = pd.read_csv(
    OUTPUT_PATH,
    sep=";",
    decimal=",",
    encoding="latin1",
    dtype=str,
)
assert df_final_check.shape == df_check.shape
assert df_final_check.columns.tolist() == EXPECTED_COLUMNS
assert set(df_final_check["Cd_Exercicio"].astype(str).unique()) == expected_exercises
assert file_sha256(OUTPUT_PATH) == temporary_hash

print(
    "Carga validada com sucesso: "
    f"{df_final_check.shape[0]} linhas, "
    f"{df_final_check.shape[1]} colunas, "
    f"tamanho={temporary_size}, sha256={temporary_hash}, "
    f"caminho={OUTPUT_PATH}, backup={backup_path}."
)


Carga validada com sucesso: 17414 linhas, 24 colunas, tamanho=5022093, sha256=431b0b63577b2e00ed7e073dd5f236baa12e12211131fd8d279768e3b664a3c2, caminho=data_output/orcamento/validation/20260730_032231/orcamento_2024_2025.csv, backup=None.
